# pbzarr cohort example: per-sample d4 → per-population mean depth

End-to-end PBZ workflow using `PbzStore`:

1. Synthesize per-sample `.d4` files with `d4tools`.
2. Create an empty `.pbz` store and import the d4s as columns of a cohort `depth` track.
3. Read the track via `store.read_track(...)`, attach a `pop` label on the `sample` dim via `dt.pbz.assign_column_labels(...)`.
4. Reduce with `groupby("pop").mean("sample")` (still needs `map_over_datasets` until xarray ships DataTree.groupby, [xarray #9472](https://github.com/pydata/xarray/issues/9472)).
5. Write the reduction back to the same store as a new track.

Requires the pixi `example` env: `pixi run -e example jupyter lab`.

In [1]:
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr

import pbzarr

work = Path(tempfile.mkdtemp(prefix="pbz_example_"))
print(work)

/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_f45nikbs


## 1. Synthesize per-sample d4 files

In [2]:
samples = {
    "s1": "POP_A",
    "s2": "POP_A",
    "s3": "POP_B",
    "s4": "POP_B",
}
contig_name, contig_len = "chr1", 1000

chrom_sizes = work / "genome.sizes"
chrom_sizes.write_text(f"{contig_name}\t{contig_len}\n")

rng = np.random.default_rng(0)
d4_paths: dict[str, Path] = {}
for name in samples:
    bg = work / f"{name}.bedgraph"
    rows = [
        f"{contig_name}\t{start}\t{start + 100}\t{int(rng.integers(0, 50))}"
        for start in range(0, contig_len, 100)
    ]
    bg.write_text("\n".join(rows) + "\n")

    d4 = work / f"{name}.d4"
    subprocess.run(
        ["d4tools", "create", "-g", str(chrom_sizes), str(bg), str(d4)],
        check=True,
    )
    d4_paths[name] = d4

d4_paths

{'s1': PosixPath('/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_f45nikbs/s1.d4'),
 's2': PosixPath('/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_f45nikbs/s2.d4'),
 's3': PosixPath('/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_f45nikbs/s3.d4'),
 's4': PosixPath('/var/folders/jd/wspg2z057cn1ndxt_ctq8j6m0000gq/T/pbz_example_f45nikbs/s4.d4')}

## 2. Bootstrap a `PbzStore` and import the d4s

`PbzStore.create(...)` returns a handle. `import_d4` requires the track to exist, so we allocate it first with `create_track`.

In [3]:
store = pbzarr.PbzStore.create(
    str(work / "cohort.pbz"),
    contigs=[contig_name],
    contig_lengths=[contig_len],
)
store.create_track(
    "depth",
    dtype="int32",
    columns=list(samples.keys()),
    column_dim="sample",
)
store.import_d4(
    "depth",
    [(str(path), name) for name, path in d4_paths.items()],
)

print("tracks:", store.tracks)
print("sample labels:", store.column_labels("depth"))

tracks: ['depth']
sample labels: ['s1', 's2', 's3', 's4']


/Users/cade/dev/pbzarr-dev/pbzarr-rs/.pixi/envs/example/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## 3. Attach a `pop` label to every contig's `sample` dim

In [4]:
depth = store.read_track("depth")
labeled = depth.pbz.assign_column_labels("sample", pop=samples)
labeled

<xarray.DataTree>
Group: /
└── Group: /chr1
        Dimensions:  (position: 1000, sample: 4)
        Coordinates:
          * sample   (sample) object 32B 's1' 's2' 's3' 's4'
            pop      (sample) <U5 80B 'POP_A' 'POP_A' 'POP_B' 'POP_B'
        Dimensions without coordinates: position
        Data variables:
            depth    (position, sample) int32 16kB dask.array<chunksize=(1000, 4), meta=np.ndarray>

## 4. Reduce by population

Today, `groupby` does not auto-propagate on `xr.DataTree` ([xarray #9472](https://github.com/pydata/xarray/issues/9472)), so we use `map_over_datasets` for that single step. When xarray ships `DataTree.groupby`, this becomes one line.

In [5]:
pop_mean = labeled.map_over_datasets(
    lambda ds: xr.Dataset({"depth_pop_mean": ds["depth"].groupby("pop").mean("sample")})
    if "depth" in ds.data_vars
    else xr.Dataset()
)
pop_mean

<xarray.DataTree>
Group: /
└── Group: /chr1
        Dimensions:         (pop: 2, position: 1000)
        Coordinates:
          * pop             (pop) object 16B 'POP_A' 'POP_B'
        Dimensions without coordinates: position
        Data variables:
            depth_pop_mean  (position, pop) float64 16kB dask.array<chunksize=(1000, 1), meta=np.ndarray>

## 5. Write back to the same store

In [6]:
store.write_track(
    "depth_pop_mean",
    pop_mean,
    description="Mean depth per population (reduction of `depth` over sample)",
)

print("tracks:", store.tracks)
print("pop labels on new track:", store.column_labels("depth_pop_mean"))

tracks: ['depth', 'depth_pop_mean']
pop labels on new track: ['POP_A', 'POP_B']


/Users/cade/dev/pbzarr-dev/pbzarr-rs/.pixi/envs/example/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## 6. Verify round-trip

In [7]:
re_opened = pbzarr.PbzStore(store.path)
expected = (
    re_opened.read_track("depth")
    .pbz.assign_column_labels("sample", pop=samples)
    .map_over_datasets(
        lambda ds: xr.Dataset({"depth_pop_mean": ds["depth"].groupby("pop").mean("sample")})
        if "depth" in ds.data_vars
        else xr.Dataset()
    )
)
got = re_opened.read_track("depth_pop_mean")
np.testing.assert_allclose(
    got[contig_name]["depth_pop_mean"].values,
    expected[contig_name]["depth_pop_mean"].values,
)
print("round-trip OK")

round-trip OK
